# 面试问题：Agent Loop 怎样检测停滞与无限循环，同时不误杀有进展的轮询？

**一句话回答。** 把每步的规范化 action、参数、可见状态版本、结果类别、成本和 side-effect 级别写入 trace；当同一状态下重复同一动作、没有可验证进展且超过窗口/预算时停止、降级或人工升级。不要仅按“重复 tool name”判死循环。

本题只用 Python 标准库重建数据合同、评分、状态机和失败分支；断言只验证小型受控样例，不能替代真实模型质量、长上下文能力或线上容量压测。

**资料入口。** [AgentBench](https://arxiv.org/abs/2308.03688) 强调多轮 Agent 的环境交互评测；本例把循环保护实现为确定性控制面，而非让模型自行承诺会停止。


In [ ]:
question = "Agent Loop 停滞检测"  # 执行本行的状态、计算或校验逻辑。
assert "Agent" in question  # 执行本行的状态、计算或校验逻辑。
assert 2 * 2 == 4  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 每个 event 记录可比较的状态，而非只保存自然语言

loop guard 需要可比较的 action、参数和状态 digest。原始 observation 可很长/含敏感信息，digest 应来自允许观察的版本化字段；副作用动作还要带 idempotency key，禁止在不确定状态下盲目重试。


In [ ]:
trace = [{"tool": "search", "args": "退款条件", "state": "s1", "result": "empty", "cost": 2, "effect": "read"}, {"tool": "search", "args": "退款条件", "state": "s1", "result": "empty", "cost": 2, "effect": "read"}]  # 执行本行的状态、计算或校验逻辑。
assert len(trace) == 2  # 执行本行的状态、计算或校验逻辑。
assert all(item["effect"] == "read" for item in trace)  # 执行本行的状态、计算或校验逻辑。
assert sum(item["cost"] for item in trace) == 4  # 执行本行的状态、计算或校验逻辑。

## 2. fingerprint 必须包含参数与状态版本

只按 tool name 会误杀分页、poll 和不同 query；只按自然语言又容易受格式噪声影响。这里用 `(tool, args, state)` 作为最小指纹，生产可加入授权、输入工件版本和 normalized schema。


In [ ]:
def fingerprint(event):  # 执行本行的状态、计算或校验逻辑。
    return event["tool"], event["args"], event["state"]  # 执行本行的状态、计算或校验逻辑。
assert fingerprint(trace[0]) == ("search", "退款条件", "s1")  # 执行本行的状态、计算或校验逻辑。
assert fingerprint(trace[0]) == fingerprint(trace[1])  # 执行本行的状态、计算或校验逻辑。
assert fingerprint({**trace[1], "args": "退款时间"}) != fingerprint(trace[0])  # 执行本行的状态、计算或校验逻辑。

## 3. 停滞定义为重复指纹且没有进展

进展必须是可验证的状态变化、候选集变化、任务子目标完成或人工批准，而非模型说“我正在尝试”。同一请求的 repeated read 在同一 state 连续出现才触发；窗口可按风险等级配置。


In [ ]:
def stagnant(events, width):  # 执行本行的状态、计算或校验逻辑。
    recent = events[-width:]  # 执行本行的状态、计算或校验逻辑。
    return len(recent) == width and len({fingerprint(item) for item in recent}) == 1 and len({item["result"] for item in recent}) == 1  # 执行本行的状态、计算或校验逻辑。
assert stagnant(trace, 2)  # 执行本行的状态、计算或校验逻辑。
progress_trace = [{"tool": "poll", "args": "job1", "state": "queued", "result": "pending", "cost": 1, "effect": "read"}, {"tool": "poll", "args": "job1", "state": "running", "result": "pending", "cost": 1, "effect": "read"}]  # 执行本行的状态、计算或校验逻辑。
assert not stagnant(progress_trace, 2)  # 执行本行的状态、计算或校验逻辑。
assert not stagnant(trace, 3)  # 执行本行的状态、计算或校验逻辑。

## 4. 预算与最大步数是独立硬门槛

即使每步都略有进展，Agent 也可能耗尽 token、时间或工具配额。预算检查不能等模型主动说结束；它应在 action dispatch 前做 reserve，避免最后一次调用把系统推入不可恢复状态。


In [ ]:
def budget_exhausted(events, max_steps, max_cost):  # 执行本行的状态、计算或校验逻辑。
    return len(events) >= max_steps or sum(item["cost"] for item in events) >= max_cost  # 执行本行的状态、计算或校验逻辑。
assert budget_exhausted(trace, 2, 10)  # 执行本行的状态、计算或校验逻辑。
assert budget_exhausted(trace, 10, 4)  # 执行本行的状态、计算或校验逻辑。
assert not budget_exhausted(progress_trace, 3, 4)  # 执行本行的状态、计算或校验逻辑。

## 5. 停止策略优先返回可解释 reason

success、用户取消、审批等待、预算、停滞和工具故障的后续动作不同。把 reason 写入终端事件，才能判断是模型能力差、工具没有数据，还是策略过严；不要只返回一个模糊的 `done`。


In [ ]:
def decide(events, success, cancelled, max_steps, max_cost):  # 执行本行的状态、计算或校验逻辑。
    if success:  # 执行本行的状态、计算或校验逻辑。
        return "success"  # 执行本行的状态、计算或校验逻辑。
    if cancelled:  # 执行本行的状态、计算或校验逻辑。
        return "cancelled"  # 执行本行的状态、计算或校验逻辑。
    if budget_exhausted(events, max_steps, max_cost):  # 执行本行的状态、计算或校验逻辑。
        return "budget"  # 执行本行的状态、计算或校验逻辑。
    if stagnant(events, 2):  # 执行本行的状态、计算或校验逻辑。
        return "stagnation"  # 执行本行的状态、计算或校验逻辑。
    return "continue"  # 执行本行的状态、计算或校验逻辑。
assert decide(trace, False, False, 9, 99) == "stagnation"  # 执行本行的状态、计算或校验逻辑。
assert decide(trace, False, False, 2, 99) == "budget"  # 执行本行的状态、计算或校验逻辑。
assert decide(progress_trace, False, False, 9, 99) == "continue"  # 执行本行的状态、计算或校验逻辑。

## 6. 高风险副作用不能由 loop guard 自动重试

循环检测只能阻止更多调用，不能修复已经发生的写入。写操作要由 idempotency key、precondition 和审批绑定；不确定执行结果时转人工/查询账本，而不是用新 key 重发。


In [ ]:
write_action = {"tool": "refund", "args": "order-7", "state": "approved-v1", "effect": "write", "idempotency": "i-7"}  # 执行本行的状态、计算或校验逻辑。
def retry_mode(action, execution_known):  # 执行本行的状态、计算或校验逻辑。
    return "inspect_ledger" if action["effect"] == "write" and not execution_known else "safe_read_retry"  # 执行本行的状态、计算或校验逻辑。
assert retry_mode(write_action, False) == "inspect_ledger"  # 执行本行的状态、计算或校验逻辑。
assert retry_mode({**write_action, "effect": "read"}, False) == "safe_read_retry"  # 执行本行的状态、计算或校验逻辑。
assert write_action["idempotency"] == "i-7"  # 执行本行的状态、计算或校验逻辑。

## 7. 停滞后选择降级、澄清或人工升级

对 read-only search，可尝试改写 query、切换索引或请求用户补充约束；对高风险任务则冻结计划并呈现 trace。fallback 也消耗预算，必须与原循环隔离并记录 policy version。


In [ ]:
def recovery(reason, effect):  # 执行本行的状态、计算或校验逻辑。
    if reason == "stagnation" and effect == "read":  # 执行本行的状态、计算或校验逻辑。
        return "clarify_or_fallback"  # 执行本行的状态、计算或校验逻辑。
    return "escalate"  # 执行本行的状态、计算或校验逻辑。
assert recovery("stagnation", "read") == "clarify_or_fallback"  # 执行本行的状态、计算或校验逻辑。
assert recovery("stagnation", "write") == "escalate"  # 执行本行的状态、计算或校验逻辑。
assert recovery("budget", "read") == "escalate"  # 执行本行的状态、计算或校验逻辑。

## 8. 评测需量化误杀与漏检

回归集应包含真实循环、正常分页、状态变化 poll、失败后安全 fallback 和副作用不确定事件。指标包括 loop recall、false-stop rate、平均无效步数、预算节省、完成率和人工升级率，不能只优化“循环数下降”。


In [ ]:
labels = {"repeat_search": True, "progress_poll": False}  # 执行本行的状态、计算或校验逻辑。
predictions = {"repeat_search": stagnant(trace, 2), "progress_poll": stagnant(progress_trace, 2)}  # 执行本行的状态、计算或校验逻辑。
assert predictions == labels  # 执行本行的状态、计算或校验逻辑。
assert set(predictions) == set(labels)  # 执行本行的状态、计算或校验逻辑。
assert decide(trace, True, False, 9, 99) == "success"  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试要说明：重复工具名不等于循环，循环是“同一可见状态下重复等价动作且没有可验证进展”；并行加入最大步数/成本硬门槛、side-effect 幂等与终端 reason。真正质量要用 loop recall 与 false-stop 一起评估，避免保护策略把正常长任务杀掉。
